# 5 · Property enrichment  (run once)

Builds the schema.org **property** retrieval vocabulary that the relation-matching notebook
(`7_property_extension.ipynb`) depends on — the property-side twin of `1_schema_enrichment.ipynb`
(which does the same for *classes*). **One-time**: re-run only to refresh the definitions.

Steps:
1. **Parse** `schemaorg-current-https.jsonld` into ~1521 properties, each with its `domain → range`
   signature, `subPropertyOf` parent(s), and `inverseOf`.
2. **Enrich** every property comment with Gemini (batched, cached, interrupt-safe).
3. **Build** `schema_properties_enriched.json` — the enriched document pool (Examples / "Distinct
   from" clauses stripped from the embedded text, kept in `enriched_comment`).

Outputs (all under `schema_org_files/`): `schema_properties.json`, `enriched_prop_defs_cache.json`,
`schema_properties_enriched.json`. Mirrors nb1 field-for-field so nb7 can treat the two pools
(classes for types, properties for relations) the same way.

In [ ]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

SCHEMA_DIR = Path("schema_org_files")     # shared schema.org assets
GEMINI_MODEL = "gemini-2.5-flash"         # authors the enriched definitions

# Enrichment knobs
ENRICH_BATCH = 100                        # schema properties per Gemini call
MAX_PROPS    = None                       # set an int for a quick dry run; None = all ~1521 properties
GEMINI_PROP_DEFS_CACHE = SCHEMA_DIR / "enriched_prop_defs_cache.json"   # label -> enriched def (resumable)
print("gemini:", GEMINI_MODEL, "| schema dir:", SCHEMA_DIR)

In [ ]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object and record latency."""
    global last_latency_s
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    )
    last_latency_s = time.perf_counter() - t0
    return json.loads(resp.text)

## 1. Parse schema.org into properties with domain → range signatures

schema.org uses `schema:domainIncludes` / `schema:rangeIncludes` (a property may apply to several
domain types and accept several range types), `rdfs:subPropertyOf` for the property hierarchy, and
`schema:inverseOf` for inverse pairs. We keep all four — `domain → range` pins the intended sense the
same way the ancestor `path` does for classes.

In [20]:
import re

def plain(value) -> str:
    if isinstance(value, dict):
        value = value.get("@value", "")
    value = re.sub(r"<[^>]+>", " ", value or "")
    return re.sub(r"\s+", " ", value).strip()

def as_list(x):
    if x is None: return []
    return x if isinstance(x, list) else [x]

def short(ref):
    """{'@id': 'schema:Person'} (or 'schema:Person') -> 'Person'."""
    if isinstance(ref, dict):
        ref = ref.get("@id", "")
    return ref.split(":")[-1]

raw = json.loads((SCHEMA_DIR / "schemaorg-current-https.jsonld").read_text())
props = {}
for node in raw["@graph"]:
    if "rdf:Property" not in as_list(node.get("@type")):
        continue
    p_id = node["@id"]
    if not p_id.startswith("schema:"):
        continue
    props[p_id] = {
        "id": p_id,
        "label": plain(node.get("rdfs:label")) or p_id.split(":")[-1],
        "comment": plain(node.get("rdfs:comment")),
        "domain": sorted({short(d) for d in as_list(node.get("schema:domainIncludes"))}),
        "range":  sorted({short(r) for r in as_list(node.get("schema:rangeIncludes"))}),
        "parents": [short(p) for p in as_list(node.get("rdfs:subPropertyOf"))
                    if isinstance(p, (str, dict))],
        "inverse_of": short(node.get("schema:inverseOf")) if node.get("schema:inverseOf") else None,
    }

def signature(p):
    """'Domain/... → Range/...' — the property analogue of a class ancestor path, for HUMAN display
    and LLM prompts. ANY when open. The '→' glyph is fine for people / the LLM."""
    d = "/".join(p["domain"]) if p["domain"] else "ANY"
    r = "/".join(p["range"])  if p["range"]  else "ANY"
    return f"{d} → {r}"

def sig_phrase(p):
    """Tokenizer-friendly signature for the EMBEDDED text: spells the direction in words. The e5 /
    bert WordPiece vocab has no '→' (it becomes [UNK], carrying no signal), so 'from X to Y' encodes
    the domain → range direction lexically instead."""
    d = " or ".join(p["domain"]) if p["domain"] else "any type"
    r = " or ".join(p["range"])  if p["range"]  else "any type"
    return f"from {d} to {r}"

prop_list = list(props.values())
print(f"{len(prop_list)} schema.org properties parsed")

1521 schema.org properties parsed


In [ ]:
# Persist the parsed properties: the baseline document pool (terse schema.org comment). Same
# "label (domain -> range): comment" embedding_text shape as the enriched pool, so the two pools
# differ only in the comment text — exactly like schema_classes.json vs schema_classes_enriched.json.
schema_properties = [
    {"id": p["id"], "label": p["label"], "comment": p["comment"],
     "domain": p["domain"], "range": p["range"], "sub_property_of": p["parents"],
     "inverse_of": p["inverse_of"], "signature": signature(p),
     "embedding_text": f'{p["label"]} ({sig_phrase(p)}): {p["comment"]}'}
    for p in prop_list
]
(SCHEMA_DIR / "schema_properties.json").write_text(json.dumps(schema_properties, indent=2, ensure_ascii=False))
print(f"wrote {len(schema_properties)} properties -> schema_properties.json")

## 2. Enrich schema property comments with Gemini

Batched and cached to `schema_org_files/enriched_prop_defs_cache.json`. Interrupt-safe: re-running
skips properties already in the cache, so you only pay for what's missing. Set `MAX_PROPS` in the
config cell for a quick partial run.

In [ ]:
ENRICH_PROP_SYSTEM_PROMPT = """You are building a retrieval vocabulary for the schema.org PROPERTY (relation) system. You are given a BATCH of schema.org properties, each with its domain -> range signature (the subject types it applies to and the object types it accepts, e.g. "Organization/Person -> Person") and its original schema.org comment. A property expresses a RELATION that holds from a subject entity to an object entity. For EACH property, write one enriched definition that will later be embedded and matched against free-form relation phrases extracted from text.

Write each definition as a few compact sentences in THIS exact order and style:
1. A precise one-sentence gloss of the RELATION the property expresses, phrased so the direction is explicit (which argument is the subject and which is the object). Use the domain -> range signature to pin down the intended sense (e.g. "author" relates a CreativeWork to the Person who wrote it, not the reverse).
2. "Also called <2-5 synonyms or alternative phrasings of the relation>." (verb phrases are fine, e.g. for "author": "written by", "created by", "authored by").
3. "Examples: <3-6 concrete subject-relation-object instances>."
4. ONLY if the property is easily confused with a sibling property, add: "Distinct from <sibling>, which is <one short clause>." Otherwise omit this sentence.

Keep every definition short and information-dense: long generic prose dilutes a mean-pooled embedding. Do not restate the label verbatim as the whole gloss. Stay faithful to schema.org's intended meaning and to the relation's direction.

Return ONLY valid JSON in this schema, covering EVERY label in the batch exactly once:
{ "definitions": { "<label>": "<enriched definition>", "...": "..." } }"""

defs = json.loads(GEMINI_PROP_DEFS_CACHE.read_text()) if GEMINI_PROP_DEFS_CACHE.exists() else {}

pool = prop_list if MAX_PROPS is None else prop_list[:MAX_PROPS]
todo = [p for p in pool if p["label"] not in defs]
print(f"{len(defs)} cached | {len(todo)} to enrich | batch={ENRICH_BATCH}")

MAX_RETRIES = 5          # attempts per batch before giving up (transient errors: rate limits, timeouts)
t_start = time.perf_counter()
for i in range(0, len(todo), ENRICH_BATCH):
    batch = todo[i:i + ENRICH_BATCH]
    payload = [{"label": p["label"], "signature": signature(p), "comment": p["comment"]} for p in batch]
    bnum = i // ENRICH_BATCH

    # Retry with exponential backoff; do NOT skip. If a batch can't be enriched after MAX_RETRIES,
    # persist what we have and stop hard so no properties are silently dropped (the cache lets you
    # resume by simply re-running this cell).
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            out = gemini_json(ENRICH_PROP_SYSTEM_PROMPT, json.dumps(payload, ensure_ascii=False))
            got = out.get("definitions", {})
            break
        except Exception as e:
            if attempt == MAX_RETRIES:
                GEMINI_PROP_DEFS_CACHE.write_text(json.dumps(defs, indent=2, ensure_ascii=False))
                raise RuntimeError(
                    f"batch {bnum} failed after {MAX_RETRIES} attempts: {e}. "
                    f"{len(defs)} defs cached — re-run this cell to resume from here."
                ) from e
            wait = min(2 ** attempt, 30)
            print(f"  batch {bnum} attempt {attempt}/{MAX_RETRIES} failed ({e}); retrying in {wait}s ...")
            time.sleep(wait)

    # keep only non-empty defs for labels we actually asked about
    missing = []
    for p in batch:
        d = got.get(p["label"])
        if d and d.strip():
            defs[p["label"]] = d.strip()
        else:
            missing.append(p["label"])
    if missing:
        print(f"  batch {bnum}: model omitted {len(missing)} label(s), will retry next run: {missing[:8]}")
    GEMINI_PROP_DEFS_CACHE.write_text(json.dumps(defs, indent=2, ensure_ascii=False))
    done = min(i + ENRICH_BATCH, len(todo))
    print(f"  {len(defs)} total enriched  (+{done}/{len(todo)} this run, {last_latency_s:.1f}s/call)")

remaining = [p["label"] for p in pool if p["label"] not in defs]
print(f"done in {time.perf_counter()-t_start:.0f}s | {len(defs)}/{len(pool)} labels have a Gemini definition")
if remaining:
    print(f"{len(remaining)} still missing (model omitted them from its JSON) — re-run this cell to fill them: {remaining[:12]}")

## 3. Build the enriched document pool

In [21]:
# Build the enriched document pool. Strip the "Examples: ..." sentence and contrastive
# "Distinct from ..." clauses from the embedded text (they hurt a bi-encoder), but keep the
# full text in enriched_comment. Identical treatment to nb1's class pool.
_CONTRAST_RE = re.compile(r"\s*(?:Distinct from|Not to be confused with)[^.]*\.", re.IGNORECASE)
_EXAMPLE_RE  = re.compile(r"\s*Examples?:.*?\.(?=\s+[A-Z]|\s*$)", re.IGNORECASE)

def embedding_comment(text):
    text = _CONTRAST_RE.sub("", text)
    text = _EXAMPLE_RE.sub("", text)
    return text.strip()

enriched_props = []
for p in prop_list:
    ec = defs.get(p["label"])
    is_en = ec is not None
    comment_for_embed = embedding_comment(ec) if is_en else p["comment"]
    enriched_props.append({
        "id": p["id"], "label": p["label"], "comment": p["comment"],
        "domain": p["domain"], "range": p["range"], "sub_property_of": p["parents"],
        "inverse_of": p["inverse_of"], "signature": signature(p),
        "enriched_comment": ec or "", "enriched": is_en,
        "embedding_text": f'{p["label"]} ({sig_phrase(p)}): {comment_for_embed}',
    })

out_pool = SCHEMA_DIR / "schema_properties_enriched.json"
out_pool.write_text(json.dumps(enriched_props, indent=2, ensure_ascii=False))
print(f"{sum(p['enriched'] for p in enriched_props)}/{len(enriched_props)} enriched -> {out_pool}")
# spot check
for lbl in ["founder", "author", "birthPlace", "memberOf"]:
    m = next((p["embedding_text"] for p in enriched_props if p["label"] == lbl), None)
    if m: print("  ", m)

1521/1521 enriched -> schema_org_files/schema_properties_enriched.json
   founder (from Organization to Organization or Person): An Organization has a founder, which is the Person or Organization that established it. Also called "creator", "establisher", "originator". Examples: "The Organization's founder is 'Steve Jobs'.", "This company's founder is 'a group of scientists'.", "The founder of the startup is 'Jane Doe'."
   author (from CreativeWork or Rating to Organization or Person): A CreativeWork or Rating has an author who is the Organization or Person responsible for its creation. Also called created by, written by, composed by, authored by. Rowling', A research paper has author 'Dr. Smith', The rating was given by author 'John Doe'.
   birthPlace (from Person to Place): A Person has a birthPlace which is the Place where they were born. Also called place of birth, born in, birth location.
   memberOf (from Organization or Person to MemberProgramTier or Organization or ProgramMemb